<a href="https://colab.research.google.com/github/jagan93550/Deep_learning/blob/main/Rnn01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Rnn


Objective:
The objective of this code is to build a Simple Recurrent Neural Network (RNN) model for text classification using the AG News dataset from Hugging Face. The code preprocesses news articles by cleaning the text, converting words into numerical sequences, and padding them to a fixed length. It then trains an RNN model to learn patterns in the news articles and classify each article into one of the four news categories. Finally, the model is evaluated using test data to measure its classification accuracy.

In [4]:
# Install the datasets library
!pip install datasets

# Import the dataset loader
from datasets import load_dataset

# Load the AG News dataset from Hugging Face
dataset = load_dataset("wangrongsheng/ag_news")

# Use only the training dataset
data = dataset["train"]

# Display one sample
print(data[0])

# Display total samples
print("Total Samples:", len(data))

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
Total Samples: 120000



# Step 2: Clean and Normalize the Text


In [6]:


# Import regular expression library
import re

# Create a function to clean text
def clean_text(text):
    text = text.lower()                  # Convert text to lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation, numbers and special characters
    return text                          # Return cleaned text

# Clean all text data
texts = [clean_text(item["text"]) for item in data]

# Get all labels
labels = [item["label"] for item in data]


# ================================
# Step 3: Tokenize the Strings into Numbers


In [8]:

# Import Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer(num_words=10000)

# Build vocabulary
tokenizer.fit_on_texts(texts)

# Convert text into integer sequences
sequences = tokenizer.texts_to_sequences(texts)

 Step 4: Pad and Truncate Sequences



In [9]:
# Import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Make every sequence exactly 50 words long
X = pad_sequences(sequences, maxlen=50)

step 5: convert label to categorical v alues

In [10]:
# Import one-hot encoder
from tensorflow.keras.utils import to_categorical

# Convert labels into one-hot vectors
y = to_categorical(labels)

# Split the dataset into training and testing data
from sklearn.model_selection import train_test_split

# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


Step 6: Define the RNN Model Architecture

In [19]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

# Create the model
model = Sequential()

# Define the input shape
model.add(Input(shape=(50,)))

# Add Embedding layer
model.add(Embedding(input_dim=10000, output_dim=64))

# Add SimpleRNN layer
model.add(SimpleRNN(64))

# Add Output layer
model.add(Dense(4, activation="softmax"))

# Display the model summary
model.summary()




Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 50, 64)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,516 (2.47 MB)

 Trainable params: 648,516 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

Purpose of the Model Summary

model.summary() displays the architecture of the RNN model. It shows the layers used, the output shape of each layer, the number of trainable parameters, and the total model size. This helps verify that the model is built correctly before training.

Explanation of Output
Layer	Purpose
Embedding (None, 50, 64)	Converts each word ID into a 64-dimensional dense vector, allowing the model to learn meaningful word representations. The input contains 50 words per sequence.
SimpleRNN (None, 64)	Processes the sequence one word at a time, remembers previous information, and produces a 64-feature output representing the entire sentence.
Dense (None, 4)	Uses the learned features to predict the probabilities of the 4 AG News categories (World, Sports, Business, Sci/Tech) using the Softmax activation function.

Step 7: Compile and Train the Model

In [13]:
# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model
model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_split=0.2
)

Epoch 1/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 55s 22ms/step - accuracy: 0.7943 - loss: 0.5285 - val_accuracy: 0.8940 - val_loss: 0.3423
Epoch 2/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 54s 23ms/step - accuracy: 0.9041 - loss: 0.3066 - val_accuracy: 0.8938 - val_loss: 0.3443
Epoch 3/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 77s 21ms/step - accuracy: 0.9202 - loss: 0.2530 - val_accuracy: 0.8767 - val_loss: 0.4030
Epoch 4/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 81s 20ms/step - accuracy: 0.9284 - loss: 0.2280 - val_accuracy: 0.8910 - val_loss: 0.3616
Epoch 5/5
2400/2400 ━━━━━━━━━━━━━━━━━━━━ 47s 20ms/step - accuracy: 0.9445 - loss: 0.1737 - val_accuracy: 0.8886 - val_loss: 0.3964


 Step 8: Read and Evaluate the Target Accuracy

In [14]:

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)

# Print the test accuracy
print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8895 - loss: 0.3917
Test Accuracy: 0.8895416855812073
